# LGD Model - Loss Given Default

Since the synthetic dataset does not include actual recovery/collateral data, this notebook estimates LGD using segment-based assumptions grounded in typical real-world retail lending patterns: secured loans (e.g. auto) recover more via collateral than unsecured loans (e.g. personal), and higher-balance loans are assumed to have marginally better recovery due to more aggressive collection efforts. This is a simplification, disclosed transparently, standing in for a full recovery-rate regression model that would require historical default/recovery data not available here.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/clean_loan_data.csv')
df.shape

(10000, 17)

In [ ]:
base_recovery_rate = {
    'auto': 0.70,
    'home_improvement': 0.55,
    'debt_consolidation': 0.35,
    'personal': 0.30,
}

df['base_recovery_rate'] = df['loan_purpose'].map(base_recovery_rate)

NameError: name 'base_recovery_rate' is not defined

In [ ]:
balance_percentile = df['current_balance'].rank(pct=True)
balance_adjustment = (balance_percentile - 0.5) * 0.10  # +/- up to 5 percentage points

df['adjusted_recovery_rate'] = (df['base_recovery_rate'] + balance_adjustment).clip(0.02, 0.90)

KeyError: 'base_recovery_rate'

In [ ]:
df['LGD'] = 1 - df['adjusted_recovery_rate']
df['LGD'] = df['LGD'].round(3)

df[['loan_id', 'loan_purpose', 'current_balance', 'adjusted_recovery_rate', 'LGD']].head(10)

,loan_id,loan_purpose,current_balance,adjusted_recovery_rate,LGD
0,1,debt_consolidation,1410.48,0.11318,0.887
1,2,personal,7287.52,0.10780,0.892
2,3,auto,20183.16,0.59131,0.409
3,4,debt_consolidation,1598.14,0.11515,0.885
4,5,auto,3752.12,0.53498,0.465
5,6,debt_consolidation,11193.45,0.17395,0.826
6,7,debt_consolidation,4537.56,0.14056,0.859
7,8,personal,12763.92,0.12859,0.871
8,9,home_improvement,16035.77,0.38583,0.614
9,10,home_improvement,2991.09,0.32863,0.671


In [ ]:
df.groupby('loan_purpose')['LGD'].mean().sort_values()

loan_purpose
auto                  0.449731
home_improvement      0.650231
debt_consolidation    0.849258
personal              0.900763
Name: LGD, dtype: float64

In [ ]:
df[['loan_id', 'LGD', 'adjusted_recovery_rate']].to_csv('../data/processed/lgd_estimates.csv', index=False)
print("Saved.")

Saved.
